# 04 Deterministic vs Probabilistic

Direct comparison of deterministic and probabilistic archetypal autoencoders.

In [ ]:
from pathlib import Path
import os
import sys
import json

def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'cytof_archetypes').exists():
            return path
    fallback = Path('/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv')
    if (fallback / 'src' / 'cytof_archetypes').exists():
        return fallback
    raise RuntimeError('Could not locate repository root containing src/cytof_archetypes')

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
def _resolve_out_dir() -> Path:
    env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
    if env:
        return Path(env)
    cfg_path = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
    if cfg_path.exists():
        try:
            import yaml
            cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
            out_raw = cfg.get('output_dir')
            if out_raw:
                out_path = Path(out_raw)
                return out_path if out_path.is_absolute() else REPO_ROOT / out_path
        except Exception:
            pass
    return REPO_ROOT / 'outputs' / 'experiment_suite'

OUT_DIR = _resolve_out_dir()

def _artifact_exists(path: Path) -> bool:
    if path.exists():
        return True
    print(f'Missing artifact: {path}')
    print('Run the suite first: python scripts/run_experiment_suite.py --config configs/experiment_suite.yaml')
    return False
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Repo root:', REPO_ROOT)
print('Using src dir:', SRC_DIR)
print('Using suite output dir:', OUT_DIR)


In [ ]:
import pandas as pd
comp_path = OUT_DIR / 'tables' / 'deterministic_vs_probabilistic_summary.csv'
if _artifact_exists(comp_path):
    comp = pd.read_csv(comp_path)
    display(comp.sort_values(['k','seed']).head(20))
else:
    comp = None

In [ ]:
if comp is not None:
    cols = ['det_test_mse','prob_test_mse','det_rare_class_error','prob_rare_class_error','wilcoxon_p','wilcoxon_q']
    display(comp[cols].describe())
    print('Probability model wins (lower test MSE):', (comp['prob_test_mse'] < comp['det_test_mse']).mean())